In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH50=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH50_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH50_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH50[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [2]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 0 < x < 100]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=50, sigma=5, gamma=1, norm2=1, mu2=50, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (25, 75)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 4862 (χ²/ndof = 113.1)     │              Nfcn = 927              │
│ EDM = 1.04e-05 (Goal: 0.0002)    │            time = 0.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.557   │   0.004   │            │            │         │         │       │
│ 1 │ mu     │   51.79   │   0.05    │            │            │   25    │   75    │       │
│ 2 │ sigma  │   5.51    │   0.06    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │   1.78    │   0.05    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.56    │   0.07    │            │            │         │         │       │
│ 5 │ mu2    │  50.412   │   0.007   │            │            │         │         │       │
│ 6 │ sigma2 │   0.59    │   0.04    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────────────────────┐
│        │      norm        mu     sigma     gamma     norm2       mu2    sigma2    gamma2 │
├────────┼─────────────────────────────────────────────────────────────────────────────────┤
│   norm │  1.46e-05  0.004e-3 -0.058e-3  0.022e-3 -0.004e-3 -0.001e-3 -0.001e-3         0 │
│     mu │  0.004e-3    0.0021   -0.0008    0.0002   -0.0000  -0.01e-3   -0.0000    0.0000 │
│  sigma │ -0.058e-3   -0.0008   0.00415   -0.0024     0.000   0.01e-3    0.0000     0.000 │
│  gamma │  0.022e-3    0.0002   -0.0024   0.00281   -0.0000        -0    0.0000    0.0000 │
│  norm2 │ -0.004e-3   -0.0000     0.000   -0.0000   0.00444   0.45e-3   -0.0026     0.000 │
│    mu2 │ -0.001e-3  -0.01e-3   0.01e-3        -0   0.45e-3   5.4e-05  -0.26e-3         0 │
│ sigma2 │ -0.001e-3   -0.0000    0.0000    0.0000   -0.0026  -0.26e-3   0.00151    0.0000 │
│ gamma2 │         0    0.0000     0.000    0.0000     0.000         0    0.0000         0 │
└────────┴─────────────────────────────────────────────────────────────────────────────────┘

In [3]:
fit_MH50_values={}
fit_MH50_errors={}

fit_values={'MH50': fit_MH50_values,}
fit_errors={'MH50_errors': fit_MH50_errors}



for param in m_voigt.parameters:
    fit_MH50_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH50_errors[error] = m_voigt.errors[error]

print(fit_MH50_values)
print(fit_MH50_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH50"]=fit_MH50_values
results["MH50_errors"]=fit_MH50_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH50"]=fit_MH50_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH50_errors"]=fit_MH50_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.5570171666241786, 'mu': 51.789847578340705, 'sigma': 5.505738056673616, 'gamma': 1.7833280320442622, 'norm2': 0.5550792018084296, 'mu2': 50.41249139360733, 'sigma2': 0.5867923616018343, 'gamma2': 0.001}
{'norm': 0.0038244086675260436, 'mu': 0.04584217566701909, 'sigma': 0.06439548050787058, 'gamma': 0.05304600281301752, 'norm2': 0.06660698434396901, 'mu2': 0.00734987539934505, 'sigma2': 0.03882543884848561, 'gamma2': 1e-05}
